#**Meeting Scheduler — MCP Demo**

An MCP (Model Context Protocol) server exposes two tools — `check_availability` and `book_meeting` — over a small mocked calendar. A chat loop (and a Gradio UI) uses OpenAI function calling to decide when to call these tools, and MCP standardizes how the model discovers and calls them.

This recreates the "Book a meeting with Raj tomorrow at 10 AM" example already drawn on the Module 5 slides — live.

Unlike a typical MCP setup (a server process + a separate client process), everything here runs in one notebook: the `fastmcp` client connects directly to the in-process `FastMCP` server object instead of over a network transport. Same protocol, no background process to manage.

###**Install Dependencies**

In [ ]:
!pip install fastmcp openai gradio

##**1. Set your OpenAI API key**

In [ ]:
# Retrieve the API key from Colab's secrets
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

##**2. Define the MCP server**
A small mocked calendar and two tools. Kept in-memory on purpose — this is a live demo, not a real scheduling system.

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP("meeting_scheduler")

WORK_HOURS = ["09:00", "10:00", "11:00", "12:00", "14:00", "15:00", "16:00"]

# person -> day -> list of already-booked "HH:MM" slots
_calendar = {
    "Raj": {"today": ["10:00", "14:00"], "tomorrow": ["09:00"]},
    "Priya": {"today": ["11:00"], "tomorrow": ["10:00", "15:00"]},
    "Karan": {"today": [], "tomorrow": ["11:00", "12:00"]},
    "Neha": {"today": ["09:00", "16:00"], "tomorrow": []},
}


def _resolve_day(day: str) -> str:
    return "tomorrow" if "tomorrow" in day.strip().lower() else "today"

In [ ]:
@mcp.tool()
def check_availability(person: str, day: str) -> str:
    """Check a colleague's free meeting slots for 'today' or 'tomorrow'."""
    person = person.strip().title()
    day_key = _resolve_day(day)

    if person not in _calendar:
        return f"I don't have a calendar for {person}. Known colleagues: {', '.join(_calendar)}."

    busy = _calendar[person][day_key]
    free = [t for t in WORK_HOURS if t not in busy]

    if not free:
        return f"{person} has no free slots {day_key}."
    return f"{person} is free {day_key} at: {', '.join(free)}"

In [ ]:
@mcp.tool()
def book_meeting(person: str, day: str, time: str, purpose: str = "") -> str:
    """Book a meeting with a colleague at a given time on 'today' or 'tomorrow'."""
    person = person.strip().title()
    day_key = _resolve_day(day)
    time = time.strip()

    if person not in _calendar:
        return f"I don't have a calendar for {person}. Known colleagues: {', '.join(_calendar)}."
    if time not in WORK_HOURS:
        return f"{time} isn't a valid slot. Working hours are: {', '.join(WORK_HOURS)}."
    if time in _calendar[person][day_key]:
        return f"{person} is already booked at {time} {day_key}. Try a different time."

    _calendar[person][day_key].append(time)
    purpose_text = f' for "{purpose}"' if purpose else ""
    return f"Booked: meeting with {person} {day_key} at {time}{purpose_text}. Confirmation sent."

##**3. Connect an MCP client and wire up OpenAI function calling**
`process_query` lists the tools MCP exposes, hands their schemas to OpenAI as function-calling tools, and — if the model decides to use one — calls it through the MCP client and feeds the result back for a final answer.

In [ ]:
import json
from openai import OpenAI
from fastmcp.client import Client

openai_client = OpenAI()


async def list_mcp_tools():
    async with Client(mcp) as client:
        tools = await client.list_tools()
        return [
            {"name": t.name, "description": t.description, "parameters": t.input_schema}
            for t in tools
        ]


async def process_query(query: str):
    async with Client(mcp) as client:
        tools = await client.list_tools()
        tool_specs = [
            {
                "type": "function",
                "function": {
                    "name": t.name,
                    "description": t.description,
                    "parameters": t.input_schema,
                },
            }
            for t in tools
        ]

        messages = [{"role": "user", "content": query}]
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tool_specs,
        )
        choice = response.choices[0].message
        messages.append(choice)

        tool_used_names = []
        tool_outputs = []

        if choice.tool_calls:
            for tool_call in choice.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)
                tool_result = await client.call_tool(fn_name, fn_args)
                tool_result_text = str(tool_result.data)

                tool_used_names.append(fn_name)
                tool_outputs.append(tool_result_text)

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_result_text,
                })

        final_response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
        )
        result = final_response.choices[0].message.content

        return (
            result,
            ", ".join(tool_used_names) if tool_used_names else "No tool used",
            "\n\n".join(tool_outputs) if tool_outputs else "N/A",
        )

##**4. Try it directly**
The exact example from the Module 5 slide — this one succeeds, since 10:00 is open on Raj's mock calendar for "tomorrow".

In [ ]:
answer, tool_used, tool_output = await process_query("Book a meeting with Raj tomorrow at 10 AM")
print("Answer:", answer)
print("Tool used:", tool_used)
print("Tool output:", tool_output)

Try a slot that's already taken, to see the model handle a conflict:

In [ ]:
answer, tool_used, tool_output = await process_query("Book a meeting with Priya tomorrow at 10 AM")
print("Answer:", answer)
print("Tool used:", tool_used)
print("Tool output:", tool_output)

##**5. Gradio UI**
Same logic, wrapped in a chat-style front end. The tool list loads from the live MCP connection, and each answer shows which tool ran and what it returned.

In [ ]:
import gradio as gr


async def show_tool_list():
    tools = await list_mcp_tools()
    html = "<ul style='padding-left: 1.2em;'>"
    for tool in tools:
        html += f"<li>🛠️ <b>{tool['name']}</b>: {tool['description']}</li>"
    html += "</ul>"
    return html


with gr.Blocks() as demo:
    gr.Markdown("# MCP + OpenAI Meeting Scheduler")
    gr.Markdown("Chat with a scheduling assistant backed by tools exposed via MCP, using OpenAI function calling.")

    tool_display = gr.HTML()
    demo.load(fn=show_tool_list, inputs=None, outputs=tool_display)

    with gr.Row():
        user_input = gr.Textbox(
            lines=2,
            placeholder="e.g. Book a meeting with Raj tomorrow at 10 AM",
            label="Your Request",
        )
        output = gr.Textbox(label="Final Answer")

    with gr.Row():
        tool_used = gr.Textbox(label="Tool Used")
        tool_output = gr.Textbox(label="Tool Output")

    submit_btn = gr.Button("Submit")
    submit_btn.click(fn=process_query, inputs=user_input, outputs=[output, tool_used, tool_output])

demo.launch()